In [ ]:
# Cell 1: Install & import libraries
!pip install requests beautifulsoup4 pandas lxml tabula-py pdfminer.six tqdm

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re, io, json, time
from tqdm import tqdm

In [2]:
# Cell 2: Define the comprehensive sources from your original app
sources = {
    'nfhs5_data': {
        'https://doi.org/10.1186/s12889-024-18784-4': 'NFHS-5 Data (Headcount/Table 2 - BMC Public Health)',
        'https://doi.org/10.1371/journal.pone.0305205': 'NFHS-5 Data (Spatial Clustering/Hotspots - PLoS ONE)',
    },
    'treatment_patterns': {
        'https://www.grandviewresearch.com/industry-analysis/india-glp-1-receptor-agonist-market-report': 'GLP-1 Market Growth Rate (CAGR 2025-2030)',
        'https://m.economictimes.com/industry/healthcare/biotech/pharmaceuticals/a-big-fat-fight-has-just-broken-out-in-india/articleshow/122049705.cms': 'GLP-1 Anti-Obesity Drug Market Value (March 2025) & Pricing',
        'https://www.iosrjournals.org/iosr-jpbs/papers/Vol19-issue6/Ser-2/L1906027179.pdf': 'GLP-1 Patient Openness/Barriers (77.3% Openness)',
        'https://nobesity.in/weight-loss-surgery-cost-in-india/': 'Bariatric Surgery Cost Range (Lakhs)',
        'https://www.frontiersin.org/journals/endocrinology/articles/10.3389/fendo.2024.1382814/full': 'Lifestyle Intervention (Clinical Basis)',
    }
}



In [3]:
# Cell 3: Extract NFHS-5 tables dynamically
nfhs_tables = []

for url, desc in sources['nfhs5_data'].items():
    print(f"\n🔗 {desc}\n{url}")
    try:
        res = requests.get(url, timeout=20)
        res.raise_for_status()
        tables = pd.read_html(res.text)
        print(f"✅ Found {len(tables)} tables")

        for t in tables:
            if any(re.search(r'obese|bmi|overweight', str(c), re.I) for c in t.columns):
                nfhs_tables.append({'source': url, 'table': t})
                display(t.head())
    except Exception as e:
        print(f"❌ Error parsing {url}: {e}")



🔗 NFHS-5 Data (Headcount/Table 2 - BMC Public Health)
https://doi.org/10.1186/s12889-024-18784-4


C:\Users\HP\AppData\Local\Temp\ipykernel_11888\3143599539.py:9: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(res.text)


❌ Error parsing https://doi.org/10.1186/s12889-024-18784-4: Missing optional dependency 'html5lib'.  Use pip or conda to install html5lib.

🔗 NFHS-5 Data (Spatial Clustering/Hotspots - PLoS ONE)
https://doi.org/10.1371/journal.pone.0305205
❌ Error parsing https://doi.org/10.1371/journal.pone.0305205: Missing optional dependency 'html5lib'.  Use pip or conda to install html5lib.


In [12]:
# couldnt get the tavle extracted from url so we donwloaded the pdf and continued the process
import camelot
import pandas as pd

pdf_path = r"C:\Users\HP\Desktop\wegovy\prevelance.pdf"

# Try stream mode (better for tables without lines)
tables = camelot.read_pdf(pdf_path, pages='all', flavor='stream')

print(f"✅ Extracted {len(tables)} tables using stream mode")

for i, t in enumerate(tables):
    print(f"\n--- Table {i+1} ---")
    print(t.df.head(10))  # show first few rows


D:\anoconda\envs\newenv\Lib\site-packages\camelot\parsers\base.py:238: UserWarning: No tables found in table area (53.12209632999997, 234.6434021, 544.1420963300002, 575.308619744009)
  cols, rows, v_s, h_s = self._generate_columns_and_rows(bbox, user_cols)
D:\anoconda\envs\newenv\Lib\site-packages\camelot\parsers\base.py:238: UserWarning: No tables found in table area (170.544998168, 42.33909985299998, 548.7986981706749, 168.78634801500897)
  cols, rows, v_s, h_s = self._generate_columns_and_rows(bbox, user_cols)
D:\anoconda\envs\newenv\Lib\site-packages\camelot\parsers\base.py:238: UserWarning: No tables found in table area (52.692901610999996, 149.006393432, 540.9257016308322, 268.4828936668485)
  cols, rows, v_s, h_s = self._generate_columns_and_rows(bbox, user_cols)
D:\anoconda\envs\newenv\Lib\site-packages\camelot\parsers\base.py:238: UserWarning: No tables found in table area (52.692897883000015, 154.870803833, 543.541697876896, 267.67200378525)
  cols, rows, v_s, h_s = self._ge

✅ Extracted 18 tables using stream mode

--- Table 1 ---
                                                   0
0                              S. V. Subramanian1,6*
1                                           Abstract
2  Background  The problem of overweight/obesity ...
3  and middle-income countries. BMI change in Ind...
4  Methods  This repeated cross-sectional study o...
5  categories of BMI among adults (age 20–54) in ...
6  2015–2016, and 2019–2021) for 36 states/UTs. S...
7  The categories were Severely/Moderately Thin (...
8  (25.0-29.9), and Obese (≥ 30.0). We also estim...
9  and headcount burden to quantify the trend of ...

--- Table 2 ---
                                                   0
0  © The Author(s) 2024. Open Access This article...
1  permits use, sharing, adaptation, distribution...
2  original author(s) and the source, provide a l...
3  other third party material in this article are...
4  to the material. If material is not included i...
5  regulation or exceeds 

In [13]:
def extract_tables_smart(pdf_path):
    """
    Extract tables from a PDF using both lattice and stream,
    then pick the one with the most non-empty content.
    """
    import camelot, os, pandas as pd

    if not os.path.exists(pdf_path):
        print(f"❌ File not found: {pdf_path}")
        return []
    
    results = []
    for flavor in ["lattice", "stream"]:
        try:
            print(f"🔍 Trying {flavor} mode...")
            tables = camelot.read_pdf(pdf_path, pages="all", flavor=flavor)
            dfs = [t.df for t in tables if not t.df.empty]
            print(f"✅ {len(dfs)} non-empty tables found using {flavor}")
            results.extend(dfs)
        except Exception as e:
            print(f"⚠️ {flavor} failed: {e}")

    # remove duplicates by shape/content
    unique_tables = []
    seen = set()
    for df in results:
        key = (df.shape, df.head().to_string())
        if key not in seen:
            unique_tables.append(df)
            seen.add(key)

    print(f"📊 {len(unique_tables)} unique tables extracted.")
    return unique_tables


In [15]:
import pdfplumber
import pytesseract
from PIL import Image
import pandas as pd

pdf_path = r"C:\Users\HP\Desktop\wegovy\prevelance.pdf"

tables_data = []
with pdfplumber.open(pdf_path) as pdf:
    for i, page in enumerate(pdf.pages):
        print(f"🧾 OCR scanning page {i+1}...")
        im = page.to_image(resolution=300)
        text = pytesseract.image_to_string(im.original)
        print(text[:500])  # preview extracted text


🧾 OCR scanning page 1...
Sung et al. BMC PublicHealth (2024) 24:1322 BMC Public Health
https://doi.org/10.1186/s12889-024-18784-4

, ®
Temporal change in prevalence of BMI cz

categories in India: patterns across States
and Union territories of India, 1999-2021

Meekang Sung'®, Akhil Kumar?®, Raman Mishra’®, Bharati Kulkarni*®, Rockli Kim?°® and
S.V. Subramanian!*"®

Abstract

Background The problem of overweight/obesity often coexists with the burden of undernutrition in most low-
and middle-income countries. BMI chang
🧾 OCR scanning page 2...
Sung et al. BMC Public Health (2024) 24:1322

Background

The Sustainable Development Goal (SDG) 2 seeks to end
hunger and ensure access to safe, nutritious, and suffi-
cient food year-round by 2030. The SDG 3 aims to ensure
healthy lives and promote wellbeing for all at all ages [1].
It is important to evaluate the nutritional status to devise
effective policies to ascertain these goals. Body Mass
Index (BMI) serves as a good metric for evaluat

In [18]:

#saved to csv and exported
import pandas as pd

tables = pd.read_excel(r"C:\Users\HP\Desktop\wegovy\India_Data.xlsx")
print(tables)
#then used this data to harcode the streamlit of states table

                       State_UT  severely/moderately thin  midly thin  \
0                         India                  29412236    50021199   
1                   Maharashtra                   4230340     6131973   
2                    Tamil Nadu                    937665     1524666   
3                 Uttar Pradesh                   3008046     5880126   
4                     Karnataka                   1710735     2932258   
5                Andhra Pradesh                   1335625     2565458   
6                       Gujarat                   3282370     4212620   
7                   West Bengal                   2809375     5359771   
8                         Bihar                   3252497     5771773   
9                     Telangana                   1093889     1543634   
10                       Kerala                    398898      681062   
11                       Punjab                    286256      504282   
12                    Rajasthan                   1

In [23]:
import camelot
import pandas as pd
import os

pdf_path = r"C:\Users\HP\Desktop\wegovy\prevelance.pdf"

def extract_tables_smart(pdf_path, show_preview=True, max_rows=10):
    """
    Extract tables from a PDF using both lattice and stream,
    then print them neatly in a structured format.
    """
    if not os.path.exists(pdf_path):
        print(f"❌ File not found: {pdf_path}")
        return []
    
    results = []
    for flavor in ["lattice", "stream"]:
        try:
            print(f"🔍 Trying {flavor} mode...")
            tables = camelot.read_pdf(pdf_path, pages="all", flavor=flavor)
            dfs = [t.df for t in tables if not t.df.empty]
            print(f"✅ {len(dfs)} non-empty tables found using {flavor}")
            results.extend(dfs)
        except Exception as e:
            print(f"⚠️ {flavor} failed: {e}")

    # Remove duplicates
    unique_tables = []
    seen = set()
    for df in results:
        key = (df.shape, df.head().to_string())
        if key not in seen:
            unique_tables.append(df)
            seen.add(key)

    print(f"\n📊 {len(unique_tables)} unique tables extracted.\n")

    # Optional structured display
    if show_preview:
        for i, df in enumerate(unique_tables):
            print("=" * 80)
            print(f"📄 Table {i+1} | Shape: {df.shape[0]} rows × {df.shape[1]} cols")
            print("-" * 80)
            # Clean up empty columns/rows for display
            df = df.dropna(how="all").replace(r"^\s*$", pd.NA, regex=True).dropna(how="all", axis=1)
            if len(df) > max_rows:
                display(df.head(max_rows))
                print(f"... (showing first {max_rows} rows)")
            else:
                display(df)
    return unique_tables

# Run and view structured tables
nfhs5_tables = {
    "BMC_PublicHealth": extract_tables_smart(pdf_path),
    "PLOS_ONE": []
}


🔍 Trying lattice mode...
✅ 7 non-empty tables found using lattice
🔍 Trying stream mode...


D:\anoconda\envs\newenv\Lib\site-packages\camelot\parsers\base.py:238: UserWarning: No tables found in table area (53.12209632999997, 234.6434021, 544.1420963300002, 575.308619744009)
  cols, rows, v_s, h_s = self._generate_columns_and_rows(bbox, user_cols)
D:\anoconda\envs\newenv\Lib\site-packages\camelot\parsers\base.py:238: UserWarning: No tables found in table area (170.544998168, 42.33909985299998, 548.7986981706749, 168.78634801500897)
  cols, rows, v_s, h_s = self._generate_columns_and_rows(bbox, user_cols)
D:\anoconda\envs\newenv\Lib\site-packages\camelot\parsers\base.py:238: UserWarning: No tables found in table area (52.692901610999996, 149.006393432, 540.9257016308322, 268.4828936668485)
  cols, rows, v_s, h_s = self._generate_columns_and_rows(bbox, user_cols)
D:\anoconda\envs\newenv\Lib\site-packages\camelot\parsers\base.py:238: UserWarning: No tables found in table area (52.692897883000015, 154.870803833, 543.541697876896, 267.67200378525)
  cols, rows, v_s, h_s = self._ge

✅ 18 non-empty tables found using stream

📊 23 unique tables extracted.

📄 Table 1 | Shape: 2 rows × 2 cols
--------------------------------------------------------------------------------


""
0
1


📄 Table 2 | Shape: 5 rows × 1 cols
--------------------------------------------------------------------------------


...

... (showing first 10 rows)
📄 Table 22 | Shape: 9 rows × 2 cols
--------------------------------------------------------------------------------


,0,1
0,and develop new interventions to target ...,<NA>
1,malnutrition.,<NA>
2,Abbreviations,References
3,BMI \nBody mass index,1. \nUNDP. The SDGs In Action: United nations...
4,NFHS \nNational family health survey,2022.https:// www. undp. org/ africa/ waca/ sd...
5,UT \nUnion territories,2024.
6,SDG \nSustainable development goal,<NA>
7,LMIC \nLow- and middle- income countries,<NA>
8,DBM \nDouble burden of malnutrition,<NA>


📄 Table 23 | Shape: 72 rows × 2 cols
--------------------------------------------------------------------------------


,0,1
0,2. \nCDC. Body Mass Index: Considerations for...,24. \nInternational Institute for Population ...
1,ease control and prevention. 2021. https:// ww...,"Family Health Survey (NFHS-2), 1998–99: India...."
2,oads/ bmifo rpact ition ers. pdf. Accessed 13 ...,dhspr ogram. com/ pubs/ pdf/ frind2/ frind2. p...
3,3. WHO. The double burden of malnutrition: p...,25. \nInternational Institute for Population ...
4,Organization; 2016. https:// www. who. int/ pu...,"Family Health Survey (NFHS-3), 2005–06: India...."
5,NMH- NHD- 17.3. Accessed 13 May 2024,dhspr ogram. com/ pubs/ pdf/ frind3/ frind3- v...
6,"4. Wells JC, Sawaya AL, Wibaek R, et al. The...",May 2024.
7,tion: aetiological pathways and consequences f...,26. \nInternational Institute for Population ...
8,2020;395(10217):75–88.,"Family Health Survey (NFHS-4), 2015-16: India...."
9,"5. \nPopkin BM, Corvalan C, Grummer-Strawn LM...",ogram. com/ pubs/ pdf/ fr339/ fr339. pdf. Acce...


... (showing first 10 rows)


In [26]:
# ================================================================
# 📊 NFHS-5 PDF → Clean Obesity Data Pipeline
# ================================================================
import os, re, camelot, pandas as pd
from IPython.display import display

# ------------------------------------------------
# 1️⃣ Extract tables from PDF (both lattice & stream)
# ------------------------------------------------
pdf_path = r"C:\Users\HP\Desktop\wegovy\prevelance.pdf"

def extract_tables_smart(pdf_path, show_preview=True, max_rows=8):
    """Extract tables using both lattice & stream modes, print structured output."""
    if not os.path.exists(pdf_path):
        print(f"❌ File not found: {pdf_path}")
        return []

    results = []
    for flavor in ["lattice", "stream"]:
        try:
            print(f"🔍 Trying {flavor} mode...")
            tables = camelot.read_pdf(pdf_path, pages="all", flavor=flavor)
            dfs = [t.df for t in tables if not t.df.empty]
            print(f"✅ {len(dfs)} non-empty tables found using {flavor}")
            results.extend(dfs)
        except Exception as e:
            print(f"⚠️ {flavor} failed: {e}")

    # remove duplicates
    unique_tables, seen = [], set()
    for df in results:
        key = (df.shape, df.head().to_string())
        if key not in seen:
            unique_tables.append(df)
            seen.add(key)

    print(f"\n📊 {len(unique_tables)} unique tables extracted.\n")

    if show_preview:
        for i, df in enumerate(unique_tables):
            print("=" * 90)
            print(f"📄 Table {i+1} | Shape: {df.shape[0]} rows × {df.shape[1]} cols")
            print("-" * 90)
            df = df.dropna(how="all").replace(r"^\s*$", pd.NA, regex=True).dropna(how="all", axis=1)
            display(df.head(max_rows))
    return unique_tables


# ------------------------------------------------
# 2️⃣ Identify BMI/Obesity-related table
# ------------------------------------------------
def find_bmi_table(table_list):
    for i, df in enumerate(table_list):
        if df.astype(str).apply(lambda x: x.str.contains("Obese|BMI", case=False)).any().any():
            print(f"✅ Found BMI/Obesity table at index {i}")
            return df
    print("❌ No BMI-related table found.")
    return None


# ------------------------------------------------
# 3️⃣ Extract and clean obesity data from the table
# ------------------------------------------------
def extract_state_obesity(df):
    pattern = r"(Men|Women).*Obese|BMI"
    # detect 'State/UT' column
    state_col = next((c for c in df.columns if re.search(r"state|ut", str(c), re.IGNORECASE)), df.columns[0])
    cols = [c for c in df.columns if re.search(pattern, str(c), re.IGNORECASE)]
    if not cols:
        print("⚠️ Could not find obesity-related columns.")
        print(f"Available columns: {list(df.columns)}")
        return pd.DataFrame()
    sub = df[[state_col] + cols].copy()
    sub.columns = ["State/UT"] + [re.sub(r"[^A-Za-z_]", "_", c) for c in cols]
    sub = sub.replace(r"^\s*$", pd.NA, regex=True).dropna(how="all")
    sub["State/UT"] = sub["State/UT"].str.strip()
    print(f"✅ Extracted obesity data for {len(sub)} states/regions.")
    return sub


# ------------------------------------------------
# 🚀 Run the pipeline
# ------------------------------------------------
nfhs5_tables = {"BMC_PublicHealth": extract_tables_smart(pdf_path)}
bmc_table = find_bmi_table(nfhs5_tables["BMC_PublicHealth"])

if bmc_table is not None and not bmc_table.empty:
    obesity_df = extract_state_obesity(bmc_table)
    display(obesity_df.head())
else:
    print("⚠️ No valid BMI table found to extract obesity data.")


🔍 Trying lattice mode...
✅ 7 non-empty tables found using lattice
🔍 Trying stream mode...


D:\anoconda\envs\newenv\Lib\site-packages\camelot\parsers\base.py:238: UserWarning: No tables found in table area (53.12209632999997, 234.6434021, 544.1420963300002, 575.308619744009)
  cols, rows, v_s, h_s = self._generate_columns_and_rows(bbox, user_cols)
D:\anoconda\envs\newenv\Lib\site-packages\camelot\parsers\base.py:238: UserWarning: No tables found in table area (170.544998168, 42.33909985299998, 548.7986981706749, 168.78634801500897)
  cols, rows, v_s, h_s = self._generate_columns_and_rows(bbox, user_cols)
D:\anoconda\envs\newenv\Lib\site-packages\camelot\parsers\base.py:238: UserWarning: No tables found in table area (52.692901610999996, 149.006393432, 540.9257016308322, 268.4828936668485)
  cols, rows, v_s, h_s = self._generate_columns_and_rows(bbox, user_cols)
D:\anoconda\envs\newenv\Lib\site-packages\camelot\parsers\base.py:238: UserWarning: No tables found in table area (52.692897883000015, 154.870803833, 543.541697876896, 267.67200378525)
  cols, rows, v_s, h_s = self._ge

✅ 18 non-empty tables found using stream

📊 23 unique tables extracted.

📄 Table 1 | Shape: 2 rows × 2 cols
------------------------------------------------------------------------------------------


""
0
1


📄 Table 2 | Shape: 5 rows × 1 cols
------------------------------------------------------------------------------------------


...

,0,1
0,and develop new interventions to target ...,<NA>
1,malnutrition.,<NA>
2,Abbreviations,References
3,BMI \nBody mass index,1. \nUNDP. The SDGs In Action: United nations...
4,NFHS \nNational family health survey,2022.https:// www. undp. org/ africa/ waca/ sd...
5,UT \nUnion territories,2024.
6,SDG \nSustainable development goal,<NA>
7,LMIC \nLow- and middle- income countries,<NA>


📄 Table 23 | Shape: 72 rows × 2 cols
------------------------------------------------------------------------------------------


,0,1
0,2. \nCDC. Body Mass Index: Considerations for...,24. \nInternational Institute for Population ...
1,ease control and prevention. 2021. https:// ww...,"Family Health Survey (NFHS-2), 1998–99: India...."
2,oads/ bmifo rpact ition ers. pdf. Accessed 13 ...,dhspr ogram. com/ pubs/ pdf/ frind2/ frind2. p...
3,3. WHO. The double burden of malnutrition: p...,25. \nInternational Institute for Population ...
4,Organization; 2016. https:// www. who. int/ pu...,"Family Health Survey (NFHS-3), 2005–06: India...."
5,NMH- NHD- 17.3. Accessed 13 May 2024,dhspr ogram. com/ pubs/ pdf/ frind3/ frind3- v...
6,"4. Wells JC, Sawaya AL, Wibaek R, et al. The...",May 2024.
7,tion: aetiological pathways and consequences f...,26. \nInternational Institute for Population ...


✅ Found BMI/Obesity table at index 5
⚠️ Could not find obesity-related columns.
Available columns: [0]


""


In [27]:
# Cell 6: Plot obesity prevalence by state
import plotly.express as px

if not obesity_df.empty:
    fig = px.bar(
        obesity_df,
        x="State/UT",
        y=obesity_df.columns[1],
        title="NFHS-5 Obesity Prevalence by State (Extracted)",
        text=obesity_df.columns[1]
    )
    fig.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
    fig.update_layout(xaxis_tickangle=-45, height=500)
    fig.show()


In [58]:
# ================== Full Content Extraction from URLs (HTML + PDF) ==================

import requests
from bs4 import BeautifulSoup
import pandas as pd
import io
import sys

# Ensure html5lib is installed
try:
    import html5lib
except ImportError:
    print("html5lib not found. Install with: pip install html5lib")
    sys.exit(1)

# Optional: PDF extraction library
try:
    from PyPDF2 import PdfReader
except ImportError:
    print("PyPDF2 not found. Install with: pip install PyPDF2")
    sys.exit(1)

urls = {
    "market_growth_rate": "https://www.grandviewresearch.com/industry-analysis/glp-1-obesity-market",
    "market_value_cr": "https://www.economictimes.indiatimes.com/nonexistent-page",
    "patient_acceptance": "https://www.iosrjournals.org/papers/patient-acceptance-of-glp1.pdf",
    "bariatric_cost_range": "https://www.nobesity.in/nonexistent-page"
}

results = {}

for key, url in urls.items():
    print(f"\nProcessing {key} -> {url}")
    try:
        r = requests.get(url)
        if r.status_code != 200:
            results[key] = {"error": f"HTTP {r.status_code} - Page not found"}
            continue

        if url.endswith(".pdf"):
            # Extract full PDF text
            try:
                reader = PdfReader(io.BytesIO(r.content))
                full_text = ""
                for page in reader.pages:
                    text = page.extract_text()
                    if text:
                        full_text += text + "\n"
                results[key] = {"pdf_full_text": full_text.strip()}
            except Exception as e:
                results[key] = {"error": f"PDF extraction failed: {e}"}
        else:
            # Extract full HTML text and numeric tables
            soup = BeautifulSoup(r.content, "html.parser")

            # Full text
            full_text = soup.get_text(separator="\n").strip()

            # Extract all tables
            try:
                tables = pd.read_html(r.text)
                table_texts = []
                for idx, table in enumerate(tables):
                    table_texts.append(f"Table {idx+1}:\n{table.to_string(index=False)}")
            except ValueError:
                table_texts = []

            results[key] = {
                "html_full_text": full_text,
                "tables_text": table_texts
            }

    except Exception as e:
        results[key] = {"error": str(e)}

# Print results for inspection (you can also save to file)
for k, v in results.items():
    print(f"\n===== {k} =====")
    if "error" in v:
        print("Error:", v["error"])
    if "pdf_full_text" in v:
        print(v["pdf_full_text"][:1000], "...")  # first 1000 chars
    if "html_full_text" in v:
        print(v["html_full_text"][:1000], "...")  # first 1000 chars
    if "tables_text" in v and v["tables_text"]:
        for t in v["tables_text"]:
            print(t, "\n")



Processing market_growth_rate -> https://www.grandviewresearch.com/industry-analysis/glp-1-obesity-market


C:\Users\HP\AppData\Local\Temp\ipykernel_11888\1746315206.py:61: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(r.text)



Processing market_value_cr -> https://www.economictimes.indiatimes.com/nonexistent-page

Processing patient_acceptance -> https://www.iosrjournals.org/papers/patient-acceptance-of-glp1.pdf

Processing bariatric_cost_range -> https://www.nobesity.in/nonexistent-page

===== market_growth_rate =====
Grand View Research


















































































Grand View Research Logo
















 


 












Toggle navigation


















Reports 








Consumer Goods »


Beauty & Personal Care


Clothing, Footwear & Accessories


Consumer F&B


Electronic & Electrical


Homecare & Decor










Semiconductors & Electronics »


Display Technologies


Electronic Security Systems


Electronic Devices


Semiconductors


Sensors & Controls










Specialty & Fine Chemicals »


Catalysts and Enzymes


Food Additives and Nutricosmetics


Renewable Chemicals


Specialty and Bio-based Polymers










Food & Beverages »


Animal Fee

In [29]:
# ✅ Fixed version: Extract tables from the correct PDF file
import camelot
import os

# Correct file path
pdf_path = r"C:\Users\HP\Desktop\wegovy\prevelance.pdf"

# Ensure the file exists
if not os.path.exists(pdf_path):
    raise FileNotFoundError(f"❌ PDF file not found at: {pdf_path}")

# Extract tables using lattice mode first
print(f"📄 Extracting tables from: {pdf_path}")
tables = camelot.read_pdf(pdf_path, pages='all', flavor='lattice')

print(f"✅ Found {len(tables)} tables using lattice mode")

# If no tables found, try stream mode as fallback
if len(tables) == 0:
    print("⚠️ Lattice mode found no tables — trying stream mode...")
    tables = camelot.read_pdf(pdf_path, pages='all', flavor='stream')
    print(f"✅ Found {len(tables)} tables using stream mode")

# Display extracted tables neatly
if len(tables) > 0:
    for i, table in enumerate(tables):
        print(f"\n--- Table {i+1} | Shape: {table.df.shape} ---")
        display(table.df.head(10))
else:
    print("❌ No tables could be extracted from the PDF.")



📄 Extracting tables from: C:\Users\HP\Desktop\wegovy\prevelance.pdf
✅ Found 7 tables using lattice mode

--- Table 1 | Shape: (2, 2) ---


,0,1
0,,
1,,



--- Table 2 | Shape: (2, 2) ---


,0,1
0,,
1,,



--- Table 3 | Shape: (5, 1) ---


...

,0,1,2,3
0,,,,
1,,,,
2,,,,
3,,,,
4,,,,
5,,,,
6,,,,
7,,,,
8,,,,
9,,,,



--- Table 6 | Shape: (4, 4) ---


,0,1,2,3
0,,,,
1,,,,
2,,,,
3,,,,



--- Table 7 | Shape: (4, 4) ---


,0,1,2,3
0,,,,
1,,,,
2,,,,
3,,,,


In [60]:
# Cell 7: Build unified dataset
results = {
    "nfhs_tables_found": len(nfhs_tables),
    "nfhs_preview": nfhs_df.head().to_dict() if not nfhs_df.empty else None,
    "treatment_sources": treatment_data
}

# Display JSON summary
print(json.dumps(results, indent=2)[:2000])


{
  "nfhs_tables_found": 0,
  "nfhs_preview": null,
  "treatment_sources": [
    {
      "url": "https://www.grandviewresearch.com/industry-analysis/india-glp-1-receptor-agonist-market-report",
      "description": "GLP-1 Market Growth Rate (CAGR 2025-2030)",
      "percents_found": [
        "34.3%",
        "71.9%",
        "27%",
        "1.0",
        "2.4",
        "44%",
        "20%",
        "39%"
      ],
      "rupees_found": [],
      "excerpt": "India GLP-1 Receptor Agonist Market | Industry Report 2030 Grand View Research Logo Home Industries Consumer Goods Beauty & Personal Care Specialty & Fine Chemicals Food & Beverages Advanced Materials Explore All Industries Companies Basic Materials Consumer Defensive Energy Financial Services Healthcare Industrials Real Estate Technology Utilities Explore All Companies Services Astra (ESG Solutio..."
    },
    {
      "url": "https://m.economictimes.com/industry/healthcare/biotech/pharmaceuticals/a-big-fat-fight-has-just-broken-ou

In [31]:
# Cell 4: Extract and parse numeric info from treatment-related articles

treatment_data = []

for url, desc in tqdm(sources['treatment_patterns'].items()):
    print(f"\n🔗 {desc}\n{url}")
    try:
        r = requests.get(url, timeout=20)
        text = BeautifulSoup(r.text, "html.parser").get_text(" ", strip=True)
        
        percents = re.findall(r"\b\d{1,2}\.\d%|\b\d{1,2}%|\b\d{1,3}\.\d\b", text)
        rupees = re.findall(r"₹\s?\d+[.,]?\d*\s?(?:crore|lakh|million)?", text, re.I)
        
        treatment_data.append({
            "url": url,
            "description": desc,
            "percents_found": list(set(percents))[:10],
            "rupees_found": list(set(rupees))[:10],
            "excerpt": text[:400] + "..."
        })
    except Exception as e:
        print(f"❌ Error scraping {url}: {e}")


  0%|                                                                                            | 0/5 [00:00<?, ?it/s]


🔗 GLP-1 Market Growth Rate (CAGR 2025-2030)
https://www.grandviewresearch.com/industry-analysis/india-glp-1-receptor-agonist-market-report


 20%|████████████████▊                                                                   | 1/5 [00:02<00:08,  2.21s/it]


🔗 GLP-1 Anti-Obesity Drug Market Value (March 2025) & Pricing
https://m.economictimes.com/industry/healthcare/biotech/pharmaceuticals/a-big-fat-fight-has-just-broken-out-in-india/articleshow/122049705.cms


 40%|█████████████████████████████████▌                                                  | 2/5 [00:04<00:07,  2.34s/it]

...

 60%|██████████████████████████████████████████████████▍                                 | 3/5 [00:14<00:11,  5.96s/it]


🔗 Bariatric Surgery Cost Range (Lakhs)
https://nobesity.in/weight-loss-surgery-cost-in-india/


 80%|███████████████████████████████████████████████████████████████████▏                | 4/5 [00:18<00:04,  4.89s/it]


🔗 Lifestyle Intervention (Clinical Basis)
https://www.frontiersin.org/journals/endocrinology/articles/10.3389/fendo.2024.1382814/full


100%|████████████████████████████████████████████████████████████████████████████████████| 5/5 [00:22<00:00,  4.47s/it]


In [32]:
# Cell 5: Show what we found
for item in treatment_data:
    print(f"\n📘 {item['description']}")
    print(f"🔗 {item['url']}")
    print(f"Percentages found: {item['percents_found']}")
    print(f"Currency mentions: {item['rupees_found']}")
    print("Excerpt:", item['excerpt'][:250], "...\n")



📘 GLP-1 Market Growth Rate (CAGR 2025-2030)
🔗 https://www.grandviewresearch.com/industry-analysis/india-glp-1-receptor-agonist-market-report
Percentages found: ['34.3%', '71.9%', '27%', '1.0', '2.4', '44%', '20%', '39%']
Currency mentions: []
Excerpt: India GLP-1 Receptor Agonist Market | Industry Report 2030 Grand View Research Logo Home Industries Consumer Goods Beauty & Personal Care Specialty & Fine Chemicals Food & Beverages Advanced Materials Explore All Industries Companies Basic Materials  ...


📘 GLP-1 Anti-Obesity Drug Market Value (March 2025) & Pricing
🔗 https://m.economictimes.com/industry/healthcare/biotech/pharmaceuticals/a-big-fat-fight-has-just-broken-out-in-india/articleshow/122049705.cms
Percentages found: ['74.2', '0.4%', '77%', '50%', '9.1%', '322.0', '357.6', '3.6%', '97%', '62%']
Currency mentions: ['₹49,999 ', '₹ 1749 ', '₹ 49 ']
Excerpt: A big fat fight has just broken out in India - The Economic Times Benchmarks Nifty 25,169.50 -32.85 FEATURED FUNDS ★★★★★ UTI